# Lab 02｜正在巡检时，告警怎样改变后续工作？

观察 `steer()` 把告警交给正在运行的 Turn，Agent 如何接着调查、再恢复巡检；另开一次任务观察 `interrupt()`。

## 核心代码：向同一 Turn 补充告警，与中断任务作对照

> 本讲重点看下面这 7 行。它们摘自后面的代码单元格，仅供定位；运行时仍从公共准备开始按顺序执行。单元格从标题所在的第 1 格起计数，行号按格内源码计算。

**第 2 节，第 7 个单元格，第 19 至 23 行：**

```python
steer_response = turn.steer(ALERT)
after_steer = len(patrol_trace.snapshot())
if steer_response.turn_id != turn.id:
    raise RuntimeError("告警没有进入原 Turn。")
patrol_status = patrol_trace.join(480)
```

`steer(ALERT)` 向正在运行的 Turn 追加输入，随后核对返回的 Turn 编号，并等待本次任务结束。

**第 4 节，第 11 个单元格，第 12 至 13 行：**

```python
turn2.interrupt()
interrupt_status = interrupt_trace.join(60)
```

`interrupt()` 请求中断，`join(60)` 等待并读取实际结束状态；两者共同区分发出请求与任务已停止。


## 公共准备｜必跑，代码可折叠

选择 **Python 3.12** 内核，保留仓库的 `examples/notebook_support.py`。下格会自动补装固定版本的实验依赖；如果当前内核已经导入其他版本，请安装后重启内核。无需先到 README 手动安装。

九讲统一读取 `DEEPSEEK_API_KEY`；未设置时弹出隐藏输入框。模型通过原生 Responses 使用 DeepSeek Flash，不需要 OpenAI / ChatGPT 账号；开始任务后会产生 API 费用。本讲准备区还执行一次简短模型连通检查，会产生少量费用。每讲可从本格独立启动，最后一格关闭连接；再次从头运行前先执行清理。

版本检查、请求观察和退出处理属于辅助代码，不必逐行阅读。主流程以“必读必跑”标注；“扩展/可选”默认跳过。


In [ ]:
import sys, os, json, secrets, time, subprocess, getpass
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
from IPython.display import display, Markdown, HTML
from html import escape
from contextlib import redirect_stdout
from io import StringIO

if sys.version_info[:2] != (3, 12):
    raise RuntimeError("请选择 Python 3.12 内核，再从头运行。")
required = {"openai-codex": "0.154.0"}
missing = []
for package, expected in required.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    if installed != expected:
        missing.append(package + "==" + expected)
if missing:
    already_loaded = any(name in sys.modules for name in ("openai_codex",))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    if already_loaded:
        raise RuntimeError("依赖已安装；请重启内核并重新运行，避免混用旧模块。")

here = Path.cwd().resolve()
examples = next((p for base in (here, *here.parents)
                 for p in (base, base / "examples") if (p / "notebook_support.py").is_file()), None)
if examples is None:
    raise FileNotFoundError("请在完整仓库内打开本 notebook。")
sys.path.insert(0, str(examples))
from notebook_support import Lab, command_receipts, tool_output_contains
from openai_codex import ApprovalMode, Sandbox
if not os.environ.get("DEEPSEEK_API_KEY", "").strip():
    os.environ["DEEPSEEK_API_KEY"] = getpass.getpass("DEEPSEEK_API_KEY：").strip()
if "lab" in globals() and not globals()["lab"]._closed:
    raise RuntimeError("本讲连接仍在运行，请先执行最后的 lab.close()。")
lab = Lab(examples / "02-instruction-control")

# 展示函数只整理真实材料，不把预期结果冒充本次运行结果。
def table(headers, rows):
    head = "".join(f"<th>{escape(str(v))}</th>" for v in headers)
    body = "".join("<tr>" + "".join(
        "<td><pre style='white-space:pre-wrap;max-width:52em'>" + escape(lab.clean(v)) + "</pre></td>"
        for v in row) + "</tr>" for row in rows)
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))

def details(title, value):
    display(HTML("<details><summary>" + escape(title) + "</summary><pre style='white-space:pre-wrap'>"
                 + escape(lab.clean(value)) + "</pre></details>"))

def run(thread, prompt, **kwargs):
    # 保留运行错误；仅收起辅助层重复打印的状态和回答。
    with redirect_stdout(StringIO()):
        return lab.run(thread, prompt, **kwargs)

def events(result):
    rows = []
    for wrapped in result.items:
        item = wrapped.root
        if item.type == "commandExecution":
            rows.extend([("采取的动作", item.command),
                         ("刚获得的信息", item.aggregated_output or "（无文本输出）")])
        elif item.type == "fileChange":
            rows.append(("文件操作", ", ".join(change.path for change in item.changes)))
        elif item.type == "agentMessage" and item.text.strip():
            rows.append(("Agent 的判断", item.text))
    table(["事件", "本次实际内容"], rows)

def snapshot(folder):
    return {str(p.relative_to(folder)):p.read_bytes() for p in folder.rglob("*") if p.is_file()}

def compare_inputs(folder, before):
    after = snapshot(folder)
    changed = [name for name in sorted(before.keys() | after.keys()) if before.get(name) != after.get(name)]
    print("输入文件：" + ("有变化，展开下方查看" if changed else "未改变"))
    for name in changed:
        details(name, "运行前：\n" + before.get(name,"[不存在]".encode()).decode(errors="replace")
                + "\n运行后：\n" + after.get(name,"[不存在]".encode()).decode(errors="replace"))



sys.path.insert(0, str(examples / "02-instruction-control"))
from control_validation import TurnTrace, successful_reads
WORK = lab.work
try:
    run(lab.thread(), "只回复连接正常，不调用工具。")
except Exception:
    lab.close()
    raise
FILES = {'servers/server-17/deployment.md': '# server-17 部署信息\n\n- 业务：payment-api（支付服务），版本 v3.8.2\n- 实例：10.0.1.17\n- 当日变更：01:40 执行变更单 CHG-2317，将数据库连接池 max_connections 由 200 下调到 60\n- 负责人：支付组\n', 'servers/server-17/metrics.md': '# server-17 监控（近 30 分钟）\n\n- payment-api P95 延迟：01:50 起由 180ms 升至 3.6s\n- 5xx 错误率：升至 8.4%\n- 数据库连接获取：大量请求等待连接，多次等待超时\n', 'servers/server-31/deployment.md': '# server-31 部署信息\n\n- 业务：order-api（订单服务），版本 v5.2.0\n- 实例：10.0.3.31\n- 当日变更：无\n', 'servers/server-31/metrics.md': '# server-31 监控（近 30 分钟）\n\n- order-api P95 延迟：稳定在 120ms\n- 5xx 错误率：0.01%\n- 无异常\n', 'tools/patrol_probe.py': 'import pathlib\nimport time\n\nHERE = pathlib.Path(__file__).resolve().parent\n(HERE / ".patrol-probe-started").write_text("started", encoding="utf-8")\nfor _ in range(25):\n    time.sleep(1)\nprint("patrol probe finished")\n', 'tools/interrupt_probe.py': 'import pathlib\nimport time\n\nHERE = pathlib.Path(__file__).resolve().parent\n(HERE / ".interrupt-probe-started").write_text("started", encoding="utf-8")\nfor _ in range(40):\n    time.sleep(1)\nprint("interrupt probe finished")\n'}
for rel in FILES:
    if rel.startswith("servers/"):
        FILES[rel] += "\n材料编号：OBS-" + secrets.token_hex(12) + "\n"
    path = WORK / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(FILES[rel], encoding="utf-8")
BASELINE = snapshot(WORK / "servers")
TOOLS_BASELINE = {name:(WORK / name).read_bytes() for name in FILES if name.startswith("tools/")}

def show_patrol():
    captured = patrol_trace.snapshot()
    reads = successful_reads(captured, WORK, FILES)
    rows = []
    submitted = False
    for event in captured:
        if event["seq"] >= before_steer and not submitted:
            rows.append(("告警提交", ALERT))
            submitted = True
        if event["method"] != "item/completed":
            continue
        item = event["item"]
        if item.get("type") == "commandExecution":
            phase = "已读材料 / 巡检" if event["seq"] < before_steer else ("恢复巡检" if "server-31" in item.get("command", "") else "后续调查")
            rows.append((phase, item.get("command", "")))
        elif item.get("type") == "agentMessage":
            rows.append(("Agent 的判断", item.get("text", "")))
    if not submitted:
        rows.append(("告警提交", ALERT))
    table(["时间线（按实际事件排列）", "动作与内容"], rows)
    before = {r["path"] for r in reads if r["completed"] < before_steer}
    after = {r["path"] for r in reads if r["started"] >= after_steer}
    later = {"servers/server-31/deployment.md", "servers/server-31/metrics.md"}
    facts = {
        "告警是否进入原 Turn": steer_response.turn_id == turn.id,
        "告警前成功读取支付服务器两份材料": first_paths <= before,
        "告警后重新读取了哪些旧材料": sorted(first_paths & after),
        "告警后成功读取待巡检服务器两份材料": later <= after,
        "Turn 最终状态": patrol_status,
        "读取回执": reads,
    }
    details("扩展：回执、内部编号与检查记录", json.dumps({"观察":facts,"事件":captured},ensure_ascii=False,indent=2))
    lab._write_json("patrol-observations.json", facts)
    for name in ("incident-report.md", "patrol-report.md"):
        p = WORK / name
        display(Markdown(f"### `{name}`\n" + (p.read_text() if p.exists() else "本次未生成。")))
    compare_inputs(WORK / "servers", BASELINE)

print("公共准备完成。")


## 1. 两台服务器，一个中途到来的告警｜必读必跑

一台正在检查，另一台稍后继续。为看清材料何时读入，本例限定独立 `cat` 读取、不用管道、不回显报告，并以慢探针留出告警窗口。这些是**观察条件**，不是推荐的巡检方法。随机编号仅用于核对读取回执，无需在报告中逐字引用。


In [ ]:
PATROL_TASK = """巡检 servers/ 下两台服务器，先 server-17，再 server-31。
每台有 deployment.md 和 metrics.md。用独立 cat 命令读取该台两份材料，不使用管道或脚本读取，
不提前读取另一台。读完 server-17 后运行 tools/patrol_probe.py，等待采集结束。
把巡检结论写入 patrol-report.md，一台一段。不要修改 servers/ 或 tools/，不要变更服务。
"""
ALERT = """【实时告警 02:07】server-17 的 payment-api P95 延迟超过 3 秒，5xx 错误率升至 8.4%。
请在当前任务中优先调查，结合已掌握的部署与变更情况，把结论和文件依据写入 incident-report.md。
使用已取得的材料，不重复读取 server-17 文件，也不要用 cat 回显报告。
调查后继续完成 server-31 的巡检，写入 patrol-report.md。
"""
table(["对象", "本次角色"], [("server-17 / payment-api", "正在检查；中途收到告警"),
                            ("server-31 / order-api", "告警处理完后继续检查")])


## 2. 把告警交给活跃 Turn｜必读必跑

等到支付服务器的两份材料确实读入，且探针正在运行，再调用 `steer()`。它追加的是同一 Turn 的输入；具体怎样响应，要看后续事件。


In [ ]:
if "patrol_trace" in globals():
    raise RuntimeError("本次已启动过巡检。不要重复运行本格；按最后清理格后的说明完整重跑。")
thread = lab.codex.thread_start(
    cwd=str(WORK), sandbox=Sandbox.workspace_write, approval_mode=ApprovalMode.deny_all,
)
# 新目录也显式清除本例标记与报告，拒绝把旧状态当成本次结果。
for name in ("tools/.patrol-probe-started", "tools/.interrupt-probe-started",
             "incident-report.md", "patrol-report.md", "interrupt-report.md"):
    (WORK / name).unlink(missing_ok=True)
turn = thread.turn(PATROL_TASK, sandbox=Sandbox.workspace_write)
patrol_trace = TurnTrace(turn).start()
first_paths = {"servers/server-17/deployment.md", "servers/server-17/metrics.md"}
try:
    patrol_trace.wait_for_marker(
        WORK / "tools/.patrol-probe-started", 180,
        ready=lambda: first_paths <= {r["path"] for r in successful_reads(patrol_trace.snapshot(), WORK, FILES)},
    )
    before_steer = len(patrol_trace.snapshot())
    steer_response = turn.steer(ALERT)
    after_steer = len(patrol_trace.snapshot())
    if steer_response.turn_id != turn.id:
        raise RuntimeError("告警没有进入原 Turn。")
    patrol_status = patrol_trace.join(480)
    if patrol_status != "completed":
        raise RuntimeError(f"巡检未完成：{patrol_status}")
except BaseException:
    patrol_trace.abort()
    raise
finally:
    (lab.runtime / "patrol-events.json").write_text(
        json.dumps(patrol_trace.snapshot(), ensure_ascii=False, indent=2), encoding="utf-8",
    )


## 3. 先读时间线，再读报告｜必读必跑

从“已读材料”往后找告警提交、调查和恢复巡检。动作若没有发生，记录就会缺少对应步骤，不能用预期补齐。回执和内部检查放在折叠区。


In [ ]:
show_patrol()


## 4. 独立观察中断｜必读必跑

这里新建 Thread，不依赖前一份报告或检查结果。`interrupt()` 请求中断当前 Turn；是否已结束看返回状态，已经产生的副作用不会自动回滚。探针只留下本地观察标记，不操作业务服务。


In [ ]:
interrupt_marker = WORK / "tools/.interrupt-probe-started"
interrupt_marker.unlink(missing_ok=True)
(WORK / "interrupt-report.md").unlink(missing_ok=True)
interrupt_thread = lab.thread(sandbox=Sandbox.workspace_write)
turn2 = interrupt_thread.turn(
    "运行 tools/interrupt_probe.py，等它完整结束后把结果写入 interrupt-report.md；不要修改其他文件。",
    sandbox=Sandbox.workspace_write,
)
interrupt_trace = TurnTrace(turn2).start()
try:
    interrupt_trace.wait_for_marker(interrupt_marker, 120)
    turn2.interrupt()
    interrupt_status = interrupt_trace.join(60)
except BaseException:
    interrupt_trace.abort()
    raise
finally:
    lab._write_json("interrupt-events.json", interrupt_trace.snapshot())
table(["观察对象", "中断后的实际情况"], [
    ("当前 Turn", interrupt_status),
    ("等待结束后才应写的报告", "存在" if (WORK / "interrupt-report.md").exists() else "不存在"),
    ("业务材料", "未改变" if snapshot(WORK / "servers") == BASELINE else "发生变化"),
    ("探针脚本", "未改变" if {name:(WORK / name).read_bytes() for name in TOOLS_BASELINE} == TOOLS_BASELINE else "发生变化"),
])


## 恢复与清理｜观察结束后必跑

若要重跑，先执行下格，再从公共准备开始；新运行使用新目录。不要向旧 Turn 重复提交告警。若只想看中断，可运行公共准备后直接跳到第 4 节。

本例展示 Codex 的 `steer()` 与 `interrupt()`。正文中的 `stop` 指停止任务循环或后续调度；不同框架的具体接口语义要分别看，不能把它等同于统一的“关闭 Thread”。


In [ ]:
try:
    for name in ("patrol_trace", "interrupt_trace"):
        trace = globals().get(name)
        if trace is not None:
            trace.abort()
finally:
    lab.close()
for name in ("patrol_trace", "interrupt_trace"):
    globals().pop(name, None)
